In [1]:
import os
import json
import mlflow

mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("brain-tumor-classification")

2026/10/31 12:28:46 INFO mlflow.tracking.fluent: Experiment with name 'brain-tumor-classification' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1793449726835, experiment_id='2', last_update_time=1793449726835, lifecycle_stage='active', name='brain-tumor-classification', tags={}>

In [2]:
def log_run(run_name, params, metrics, history, files):
    with mlflow.start_run(run_name=run_name):

        for name in params:
            mlflow.log_param(name, params[name])

        for name in metrics:
            mlflow.log_metric(name, metrics[name])

        if history is not None:
            for name in history:
                for epoch in range(len(history[name])):
                    mlflow.log_metric(name, history[name][epoch], step=epoch + 1)

        for file_path, folder in files:
            if os.path.exists(file_path):
                mlflow.log_artifact(file_path, folder)
            else:
                print("File not found, skipped:", file_path)

    print(run_name, "logged")

In [3]:
common = {
    "base_model": "InceptionV3",
    "image_size": 299,
    "batch_size": 32,
    "dropout": 0.3,
    "optimizer": "Adam",
    "loss": "categorical_crossentropy",
    "max_epochs": 20,
    "early_stopping_patience": 3,
}

params = dict(common)
params["phase"] = "feature_extraction"
params["learning_rate"] = 0.001
params["trainable_layers"] = 0

metrics = {
    "train_accuracy": 0.9457,
    "test_accuracy": 0.9051,
    "test_precision_macro": 0.90,
    "test_recall_macro": 0.90,
    "test_f1_macro": 0.90,
}

files = [
    ("/app/models/classification/best_feature_extraction.keras", "model"),
    ("/app/reports/confusion_matrix_feature_extraction.png", "plots"),
    ("/app/models/classification/class_names.json", "config"),
]
log_run("feature_extraction", params, metrics, None, files)

2026/10/31 12:29:31 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet

2026/10/31 12:29:39 INFO mlflow.tracking._tracking_service.client: 🏃 View run feature_extraction at: htt

feature_extraction logged


In [4]:
params = dict(common)
params["phase"] = "fine_tuning"
params["learning_rate"] = 0.00001
params["trainable_layers"] = 35

metrics = {
    "train_accuracy": 0.9993,
    "test_accuracy": 0.9545,
    "test_precision_macro": 0.95,
    "test_recall_macro": 0.95,
    "test_f1_macro": 0.95,
    "best_val_loss": 0.1111,
}

history = json.load(open("/app/reports/history_ft_last50.json"))

files = [
    ("/app/models/classification/best_fine_tuning_last50.keras", "model"),
    ("/app/reports/confusion_matrix_fine_tuning_last50.png", "plots"),
    ("/app/reports/training_fine_tuning_last50.png", "plots"),
]
log_run("fine_tuning_last50_layers", params, metrics, history, files)

2026/10/31 12:29:58 INFO mlflow.tracking._tracking_service.client: 🏃 View run fine_tuning_last50_layers at: http://mlflow:5000/#/experiments/2/runs/edee6b4d3e704f9caa83e061957e7cd6.
2026/10/31 12:29:58 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow:5000/#/experiments/2.


fine_tuning_last50_layers logged


In [5]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

runs = mlflow.search_runs(
    experiment_names=["brain-tumor-classification"],
    order_by=["metrics.test_accuracy DESC"])
print(runs[["tags.mlflow.runName", "metrics.test_accuracy"]])

best_run_id = runs["run_id"][0]
result = mlflow.register_model("runs:/" + best_run_id + "/model", "neuroscan-classifier")
client.set_registered_model_alias("neuroscan-classifier", "production", result.version)
print("Classifier registered: version", result.version)

runs_det = mlflow.search_runs(
    experiment_names=["brain-tumor-detection"],
    order_by=["metrics.test_mAP50 DESC"])

best_det_id = runs_det["run_id"][0]
result_det = mlflow.register_model("runs:/" + best_det_id + "/model", "neuroscan-detector")
client.set_registered_model_alias("neuroscan-detector", "production", result_det.version)
print("Detector registered: version", result_det.version)

Successfully registered model 'neuroscan-classifier'.
2026/10/31 12:34:16 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: neuroscan-classifier, version 1


         tags.mlflow.runName  metrics.test_accuracy
0  fine_tuning_last50_layers                 0.9545
1         feature_extraction                 0.9051


Created version '1' of model 'neuroscan-classifier'.
Successfully registered model 'neuroscan-detector'.


Classifier registered: version 1


2026/10/31 12:34:17 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: neuroscan-detector, version 1


Detector registered: version 1


Created version '1' of model 'neuroscan-detector'.
